# FactCheck AI - Colab Training Notebook

Train a DeBERTa-based classifier and export the model for the backend.

## 1) Install dependencies
This installs training libraries for Colab.

In [1]:
!pip -q install transformers datasets accelerate evaluate scikit-learn huggingface_hub

## 2) (Optional) Mount Drive
Use Drive if you want to save model artifacts.

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not running in Colab; skipping Drive mount.", e)

Not running in Colab; skipping Drive mount. No module named 'google.colab'


## 3) Upload CSV datasets
Upload any CSV that has `title` + `text` + `label` columns, or `text` + `label`.
Labels can be `fake`/`real` or `1`/`0`.

In [3]:
try:
    from google.colab import files
    uploaded = files.upload()
except Exception as e:
    uploaded = {}
    print("Not running in Colab; upload skipped.", e)

list(uploaded.keys()) if uploaded else []

Not running in Colab; upload skipped. No module named 'google.colab'


[]

## 3b) Optional: download extra datasets (Hugging Face)
This cell pulls a couple of public datasets to broaden coverage. Adjust the sample sizes if you want more or less data.

In [4]:
from datasets import load_dataset
import pandas as pd
import io
import requests

hf_frames = []

def _normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"fake", "1", "true", "yes"}:
            return 1
        if v in {"real", "0", "false", "no"}:
            return 0
    if value in {1, 1.0, True}:
        return 1
    if value in {0, 0.0, False}:
        return 0
    try:
        return int(value)
    except Exception:
        return None

def _append_frame(df):
    df = df.dropna(subset=["combined", "label"]).copy()
    df["label"] = df["label"].astype(int)
    hf_frames.append(df[["combined", "label"]])

def _df_to_frame(df):
    if "label" not in df.columns:
        return None
    if "title" in df.columns and "text" in df.columns:
        df["combined"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
    elif "text" in df.columns:
        df["combined"] = df["text"].fillna("")
    elif "statement" in df.columns:
        df["combined"] = df["statement"].fillna("")
    else:
        return None
    df["label"] = df["label"].map(_normalize_label)
    df = df.dropna(subset=["label"])
    return df

# Hugging Face datasets (may require datasets stored in standard formats)
HF_DATASETS = [
    ("liar", "train[:20000]"),
    ("fever", "train[:20000]"),
]
if HF_DATASETS:
    for name, split in HF_DATASETS:
        try:
            ds = load_dataset(name, split=split)
            df = ds.to_pandas()
            # Custom mapping for LIAR / FEVER
            if "statement" in df.columns and "label" in df.columns:
                fake_labels = {"pants-fire", "false", "barely-true"}
                real_labels = {"half-true", "mostly-true", "true"}
                df["label"] = df["label"].map(lambda v: 1 if v in fake_labels else 0 if v in real_labels else None)
            if "claim" in df.columns and "label" in df.columns:
                df["label"] = df["label"].map({"SUPPORTS": 0, "REFUTES": 1})
                df["statement"] = df["claim"]
            df = _df_to_frame(df)
            if df is not None:
                _append_frame(df)
                print(f"Loaded {name}: {len(df)}")
            else:
                print(f"{name} skipped: unsupported columns")
        except Exception as e:
            print(f"{name} skipped:", e)
else:
    print("HF datasets: none")

# Optional URL CSVs (public, direct CSV links)
URL_CSVS = [
    # "https://example.com/your_dataset.csv",
]
if URL_CSVS:
    for url in URL_CSVS:
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            df = pd.read_csv(io.BytesIO(resp.content))
            df = _df_to_frame(df)
            if df is not None:
                _append_frame(df)
                print(f"Loaded URL: {url}")
            else:
                print(f"URL skipped (unsupported columns): {url}")
        except Exception as e:
            print(f"URL skipped: {url} -> {e}")
else:
    print("URL CSVs: none")

print("HF/URL frames:", len(hf_frames))

c:\Users\bc833\Downloads\fake-news-extension\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


liar skipped: Dataset scripts are no longer supported, but found liar.py
fever skipped: Dataset scripts are no longer supported, but found fever.py
URL CSVs: none
HF/URL frames: 0


## 4) Load and normalize data

In [5]:
import pandas as pd
import os
from pathlib import Path

def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"fake", "1", "true", "yes"}:
            return 1
        if v in {"real", "0", "false", "no"}:
            return 0
    if value in {1, 1.0, True}:
        return 1
    if value in {0, 0.0, False}:
        return 0
    try:
        return int(value)
    except Exception:
        return None

uploaded = uploaded if "uploaded" in globals() else {}
hf_frames = hf_frames if "hf_frames" in globals() else []

def _frame_from_df(df):
    if "title" in df.columns and "text" in df.columns:
        df["combined"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()
    elif "text" in df.columns:
        df["combined"] = df["text"].fillna("")
    elif "statement" in df.columns:
        df["combined"] = df["statement"].fillna("")
    else:
        return None
    df["label"] = df["label"].map(normalize_label)
    df = df.dropna(subset=["combined", "label"])
    df["label"] = df["label"].astype(int)
    return df[["combined", "label"]]

frames = []

# Load uploaded CSVs (Colab upload)
for name in uploaded.keys():
    if name.endswith(".csv"):
        df = pd.read_csv(name)
        if "label" not in df.columns:
            continue
        frame = _frame_from_df(df)
        if frame is not None:
            frames.append(frame)

# Load local CSVs if available (useful for local runs)
candidate_dirs = [Path(".").resolve(), Path(".").resolve() / "backend" / "training", Path(".").resolve() / "training"]
local_files = ["Fake.csv", "True.csv", "fake_news_dataset_44k.csv", "fake_news_dataset_20k.csv", "fake_news.csv"]
for base in candidate_dirs:
    for name in local_files:
        path = base / name
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if name in {"Fake.csv", "True.csv"}:
            df["label"] = 1 if name.lower().startswith("fake") else 0
            df["combined"] = (df.get("title", "").fillna("") + " " + df.get("text", "").fillna("")).str.strip()
            df = df.dropna(subset=["combined"])
            frames.append(df[["combined", "label"]])
        else:
            if "label" not in df.columns:
                continue
            frame = _frame_from_df(df)
            if frame is not None:
                frames.append(frame)

frames.extend(hf_frames)

if not frames:
    raise RuntimeError("No usable CSVs found. Upload data, add local CSVs, or enable HF/URL datasets above.")

data = pd.concat(frames, ignore_index=True)
data = data.drop_duplicates(subset=["combined"])
data = data[data["combined"].str.len() >= 30]
data["combined"] = data["combined"].str[:5000]
print(data["label"].value_counts())

label
0    52331
1    45370
Name: count, dtype: int64


## 4b) Optional: holdout, balance, and dedup
Creates a holdout CSV for final evaluation, balances labels, and optionally removes near-duplicates.

In [6]:
from datasets import Dataset
import numpy as np
import re
import pandas as pd

HOLDOUT_FRAC = 0.05
BALANCE_CLASSES = True
ENABLE_NEAR_DUP = False

if not isinstance(data, pd.DataFrame):
    data = data.to_pandas() if hasattr(data, "to_pandas") else pd.DataFrame(data)
if "label" not in data.columns and data.index.name == "label":
    data = data.reset_index()
if "label" not in data.columns:
    raise RuntimeError(f"Expected 'label' column in data. Columns: {list(data.columns)}")

# Holdout set for final evaluation (kept out of training)
holdout = data.sample(frac=HOLDOUT_FRAC, random_state=42)
data = data.drop(holdout.index).reset_index(drop=True)
holdout.to_csv("eval_holdout.csv", index=False)

# Balance classes by downsampling the majority class
if BALANCE_CLASSES:
    counts = data["label"].value_counts()
    min_count = counts.min()
    balanced_frames = []
    for lbl, df in data.groupby("label"):
        balanced_frames.append(df.sample(n=min_count, random_state=42))
    data = pd.concat(balanced_frames, ignore_index=True)

# Optional near-duplicate removal (requires datasketch)
if ENABLE_NEAR_DUP:
    try:
        from datasketch import MinHash, MinHashLSH
        def _shingles(text, size=5):
            text = re.sub(r"\s+", " ", text.lower()).strip()
            if len(text) <= size:
                return [text]
            return [text[i:i+size] for i in range(0, len(text)-size+1, size)]
        lsh = MinHashLSH(threshold=0.9, num_perm=64)
        to_drop = set()
        for i, text in enumerate(data["combined"]):
            m = MinHash(num_perm=64)
            for sh in _shingles(text):
                m.update(sh.encode("utf-8"))
            if lsh.query(m):
                to_drop.add(i)
            else:
                lsh.insert(i, m)
        if to_drop:
            data = data.drop(index=list(to_drop)).reset_index(drop=True)
            print("Near-duplicates removed:", len(to_drop))
    except Exception as e:
        print("Near-dup skipped:", e)

print(data["label"].value_counts())
dataset = Dataset.from_pandas(data, preserve_index=False)

label
0    43033
1    43033
Name: count, dtype: int64


## 5) Train/valid split

In [20]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset['train']
eval_ds = dataset['test']
len(train_ds), len(eval_ds)

(77459, 8607)

## 6) Tokenize and train DeBERTa

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = "/content/factcheck-deberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["combined"], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
eval_ds = eval_ds.map(tokenize, batched=True)
train_ds = train_ds.remove_columns(["combined"])
eval_ds = eval_ds.remove_columns(["combined"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = (preds == labels).mean().item()
    return {"f1_macro": f1, "accuracy": acc}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

c:\Users\bc833\Downloads\fake-news-extension\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bc833\.cache\huggingface\hub\models--microsoft--deberta-v3-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-base

ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

## 7) Evaluate and save

In [ ]:
metrics = trainer.evaluate()
metrics

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
OUTPUT_DIR

## 8) Evaluation report and confusion matrix
Saves a classification report and confusion matrix alongside metrics.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import json

pred = trainer.predict(eval_ds)
preds = np.argmax(pred.predictions, axis=-1)
labels = pred.label_ids

report = classification_report(labels, preds, target_names=["real", "fake"], output_dict=True)
cm = confusion_matrix(labels, preds)

pd.DataFrame(cm, index=["real", "fake"], columns=["pred_real", "pred_fake"]).to_csv("confusion_matrix.csv", index=True)
with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

report

## 8) Export for the backend
Option A: Copy the folder to Google Drive.
Option B: Push to Hugging Face Hub and set `DEBERTA_MODEL` in your backend env.

In [ ]:
# Option A - copy to Drive
!cp -r /content/factcheck-deberta /content/drive/MyDrive/factcheck-deberta

In [ ]:
# Option B - push to Hugging Face Hub
# from huggingface_hub import login
# login()  # enter your HF token
# trainer.push_to_hub('your-hf-username/factcheck-deberta')

## Backend integration
Set your backend env var to the model you pushed, for example:

`DEBERTA_MODEL=your-hf-username/factcheck-deberta`